In [ ]:
import os, sys
import argparse

from IPython.display import Markdown, display

display(Markdown("README.md"))

sys.path.insert(0, os.path.abspath('../../../python'))

if os.path.isfile('Makefile'):
    os.system("make clean && make distclean")

In [ ]:
import peano4
import exahype2

dimension         = 2
width             = [10,10]
offset            = [0,0]
end_time          = 0.5
plot_interval     = 0.1
patch_size        = 10
maximum_cell_size = 0.1
minimum_cell_size = 0.01
space_order       = 2
time_order        = 2

solvers = {
  "fv-rusanov-global-fixed":exahype2.solvers.fv.rusanov.GlobalFixedTimeStep,
  "fv-rusanov-global-fixed-enclave":exahype2.solvers.fv.rusanov.GlobalFixedTimeStepWithEnclaveTasking,
  "fv-rusanov-global-adaptive":exahype2.solvers.fv.rusanov.GlobalAdaptiveTimeStep,
  "fv-rusanov-global-adaptive-enclave":exahype2.solvers.fv.rusanov.GlobalAdaptiveTimeStepWithEnclaveTasking,
  #"fv-rusanov-local-enclave":exahype2.solvers.fv.rusanov.LocalTimeStepWithEnclaveTasking,
  "rkdg":exahype2.solvers.rkdg.rusanov.GlobalAdaptiveTimeStep,
}
solver="fv-rusanov-global-adaptive"

build_modes = { 
  "release":peano4.output.CompileMode.Release,
  "trace":peano4.output.CompileMode.Trace,
  "assert":peano4.output.CompileMode.Asserts,
  "stats":peano4.output.CompileMode.Stats,
  "debug":peano4.output.CompileMode.Debug,
}
build_mode = "release"

unknowns = {
  "p":1,
  "v":dimension,
}
parameters = {
  "rho":2.7,
  "c":6.0,
  "sigma":0.1149,
  "t0":0.7,
  "M0":1000.0,
}

In [ ]:
initial_conditions = """
  Q[p] = 0.0;
  Q[v + 0] = 0.0;
  Q[v + 1] = 0.0;
"""
boundary_conditions = """
  Qoutside[p] = 0.0;
  Qoutside[v + 0] = 0.0;
  Qoutside[v + 1] = 0.0;
"""
flux = """
  const double K0 = rho * c * c;
  if (normal == 0) {
    F[p] = -K0 * Q[v + 0];
    F[v + 0] = -1.0 / rho * Q[p];
    F[v + 1] = 0.0;
  } else {
    F[p] = -K0 * Q[v + 1];
    F[v + 0] = 0.0;
    F[v + 1] = -1.0 / rho * Q[p];
  }
"""
eigenvalues = """
  return c;
"""
source_term = """
  if (tarch::la::norm2(volumeCentre - tarch::la::Vector<Dimensions, double>{5, 5}) < 0.1) {
    const double f = M0 * std::exp(-((t - t0) * (t - t0)) / (2.0 * sigma * sigma));
    S[p] = f;
    S[v + 0] = 0.0;
    S[v + 1] = 0.0;
  }
"""

In [ ]:
solver_name = "AcousticWaveSolver"
my_solver = solvers[solver](
    name                      = solver_name,
    patch_size                = patch_size,
    unknowns                  = sum(unknowns.values()),
    auxiliary_variables       = 0,
    min_volume_h              = minimum_cell_size,
    max_volume_h              = maximum_cell_size,
    time_step_relaxation      = 0.01,
    #pde_terms_without_state  = True
    #rk_order                 = args.time_order,
    #polynomials              = exahype2.solvers.GaussLegendreBasis(args.space_order),
    #face_projections         = exahype2.solvers.rkdg.FaceProjections.Solution,
)

my_solver.set_implementation(
  initial_conditions   = initial_conditions,
  boundary_conditions  = boundary_conditions,
  flux                 = flux,
  eigenvalues          = eigenvalues,
  source_term          = source_term,
)

index = 0
for key, value in unknowns.items():
  my_solver.add_solver_constants("static constexpr int {} = {};".format(key, index))
  index += value

for key, value in parameters.items():
  my_solver.add_solver_constants("static constexpr double {} = {};".format(key, value))

print(my_solver)

In [ ]:
my_project = exahype2.Project( ["applications","exahype2", "acousticwave"], "AcousticWaveApplication", ".", executable="acoustic-wave-application-" + str(dimension) + "d-patch-size-" + str(patch_size) + "-cell-size-" + str(maximum_cell_size))
my_project.add_solver( my_solver )

my_project.set_load_balancing( "toolbox::loadbalancing::RecursiveSubdivision", "new ::exahype2::LoadBalancingConfiguration()" )

my_project.set_global_simulation_parameters(
  dimensions            = dimension,
  size                  = width,
  offset                = offset,
  min_end_time          = end_time,
  max_end_time          = end_time,
  first_plot_time_stamp = 0.0,
  time_in_between_plots = plot_interval,
  periodic_BC           = [False, False, False]
)

print(my_project)

In [ ]:
build_mode = build_modes[build_mode]
my_project.set_Peano4_installation("../../../", build_mode )

peano4_project = my_project.generate_Peano4_project(verbose=True)
peano4_project.generate(throw_away_data_after_generation=True)
peano4_project.build(make_clean_first=True)

print(peano4_project)

In [ ]:
peano4_project.run( executable="acoustic-wave-application-" + str(dimension) + "d-patch-size-" + str(patch_size) + "-cell-size-" + str(maximum_cell_size), pipefile="output.log" )

In [ ]:
import peano4.visualisation.render
peano4.visualisation.render.render(filename="solution-AcousticWaveSolver.peano-patch-file", verbose=True)

In [ ]:
import vtk
import pyvista
from pyvista import examples
filename = examples.download_wavy(load=False)
filename.split("/")[-1]  # omit the path
reader = pyvista.get_reader("solution-AcousticWaveSolver.pvd")
print(reader)
print(reader.time_values)  
#reader.set_active_time_point(5)
reader.active_time_value
mesh = reader.read()[0]  # MultiBlock mesh with only 1 block
print(mesh)
mesh.plot()